# ChakraTS on GIFT-Eval (hosted API)

[ChakraTS](https://huggingface.co/yhatlabs/ChakraTS) is served through an API by [YHat Labs](https://yhatlabs.com); no weights are
distributed. This notebook regenerates the submitted `results/ChakraTS/all_results.csv` by calling the API and scoring with the
leaderboard's own protocol (`gift_eval` data loading, gluonts `evaluate_forecasts`, `axis=None`, NaN labels masked, multivariate
datasets evaluated per variate).

**Setup**

1. Install this repository (`pip install -e .`), download the data and set `GIFT_EVAL` as the README describes.
2. Run this notebook from the `notebooks/` folder (it reads `dataset_properties.json` next to it).
3. Set `YHAT_API_KEY`. Evaluation keys are issued on request (founders@yhatlabs.com); the key used for the leaderboard rerun is
   restricted to the `chakra-ts-gift` model, the GIFT-Eval evaluation configuration of ChakraTS.

**Quick checks first.** `MODE = "quick"` reruns a dozen small configurations in a few minutes and prints each result next to the
submitted row; `MODE = "task"` reruns one configuration of your choice; `MODE = "full"` reruns all 97 (the end of the notebook
estimates how long that takes from the measured throughput). With the defaults below a rerun reproduces the submitted rows
up to GPU rounding (a few hundredths of a percent).

**Two rules the client follows.** A call never mixes test windows of the same series: the model derives its combination from
the context before the forecast origin, and pooling across windows would let a later window inform an earlier one. And
forecasts are handed to `evaluate_forecasts` in the order `test_data` yields them, which is how gluonts pairs them with labels.
`BATCH` matters too: the model's combination is pooled over the series in a call, and the submitted run used 100 at a time.

In [ ]:
import os

API_URL = os.environ.get("YHAT_API_URL", "https://api.yhatlabs.com/v1/forecast")
API_KEY = os.environ["YHAT_API_KEY"]
MODEL = "chakra-ts-gift"   # GIFT-Eval evaluation configuration of ChakraTS (see the model card)
MODEL_NAME = "ChakraTS"    # leaderboard name; results go to ../results/ChakraTS/all_results.csv

MODE = "quick"             # "quick" (13 small configurations, minutes) | "task" (TASK below) | "full" (all 97)
TASK = "us_births/D/short"
BATCH = 100                # series per API call; the submitted run grouped series in hundreds, so this reproduces it
MAX_CONTEXT = 20480        # points of history sent per series; enough for every component's context window
SUBMITTED = "../results/ChakraTS/all_results.csv"  # printed next to each rerun when present

In [ ]:
import json, csv, time
import numpy as np, pandas as pd, requests
from gift_eval.data import Dataset
from gluonts.model.forecast import QuantileForecast
from gluonts.model import evaluate_forecasts
from gluonts.ev.metrics import MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND, MeanWeightedSumQuantileLoss
from gluonts.time_feature import get_seasonality

QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
QKEYS = [str(q) for q in QUANTILES]
METRICS = [MSE(forecast_type="mean"), MSE(forecast_type=0.5), MAE(), MASE(), MAPE(), SMAPE(), MSIS(), RMSE(), NRMSE(), ND(), MeanWeightedSumQuantileLoss(quantile_levels=QUANTILES)]
COLUMNS = ["dataset", "model", "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]", "eval_metrics/MASE[0.5]", "eval_metrics/MAPE[0.5]",
           "eval_metrics/sMAPE[0.5]", "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]", "eval_metrics/ND[0.5]",
           "eval_metrics/mean_weighted_sum_quantile_loss", "domain", "num_variates"]

# The 97 configurations, enumerated exactly as the repository's other notebooks do.
short_datasets = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
pretty_names = {"saugeenday": "saugeen", "temperature_rain_with_missing": "temperature_rain", "kdd_cup_2018_with_missing": "kdd_cup_2018", "car_parts_with_missing": "car_parts"}
dataset_properties_map = json.load(open("dataset_properties.json"))

def config_of(ds_name, term):
    if "/" in ds_name:
        ds_key, ds_freq = ds_name.split("/"); ds_key = pretty_names.get(ds_key.lower(), ds_key.lower())
    else:
        ds_key = pretty_names.get(ds_name.lower(), ds_name.lower()); ds_freq = dataset_properties_map[ds_key]["frequency"]
    return f"{ds_key}/{ds_freq}/{term}", ds_key

ALL = []  # (ds_name, term, config, ds_key)
for ds_name in short_datasets.split():
    for term in ["short", "medium", "long"]:
        if term in ("medium", "long") and ds_name not in med_long_datasets.split(): continue
        cfg, key = config_of(ds_name, term); ALL.append((ds_name, term, cfg, key))
assert len(ALL) == 97

QUICK = ["us_births/D/short", "us_births/W/short", "us_births/M/short", "saugeen/M/short", "saugeen/W/short", "ett1/W/short", "ett1/D/short", "ett2/W/short", "ett2/D/short", "bizitobs_l2c/H/short", "bizitobs_l2c/H/long", "jena_weather/D/short", "m_dense/D/short"]
SELECTED = [x for x in ALL if (MODE == "full") or (MODE == "task" and x[2] == TASK) or (MODE == "quick" and x[2] in QUICK)]
print(f"{len(SELECTED)} configuration(s) selected")

In [ ]:
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"})

def forecast_batch(contexts, horizon, freq):
    body = {"model": MODEL, "horizon": horizon, "freq": freq,
            "series": [{"id": str(i), "values": [None if not np.isfinite(v) else float(v) for v in c]} for i, c in enumerate(contexts)]}
    for attempt in range(6):
        r = session.post(API_URL, json=body, timeout=1800)
        if r.status_code in (429, 503):  # quota window or the model starting up: wait and retry
            time.sleep(min(300, 30 * 2 ** attempt)); continue
        if not r.ok: raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
        return [np.asarray(f["quantiles"], dtype=np.float32) for f in r.json()["forecasts"]]  # each [horizon, 9]
    raise RuntimeError("the API kept answering 429/503")

def forecasts_for(dataset):
    inputs = list(dataset.test_data.input); labels = list(dataset.test_data.label)
    # group instances by test window (an item's windows are yielded oldest first), never mixing windows in one call
    seen, by_window = {}, {}
    for idx, e in enumerate(inputs):
        j = seen.get(e["item_id"], 0); seen[e["item_id"]] = j + 1; by_window.setdefault(j, []).append(idx)
    q = {}
    for j in sorted(by_window):
        for i in range(0, len(by_window[j]), BATCH):
            ids = by_window[j][i:i + BATCH]
            out = forecast_batch([np.asarray(inputs[x]["target"], dtype=np.float32)[-MAX_CONTEXT:] for x in ids], dataset.prediction_length, dataset.freq)
            for x, f in zip(ids, out): q[x] = f
    # evaluate_forecasts pairs forecasts with test instances by position
    return [QuantileForecast(forecast_arrays=np.ascontiguousarray(q[x].T), start_date=labels[x]["start"], forecast_keys=QKEYS, item_id=inputs[x]["item_id"]) for x in range(len(inputs))], len(inputs)

In [ ]:
submitted = pd.read_csv(SUBMITTED).set_index("dataset") if os.path.exists(SUBMITTED) else None
out_dir = f"../results/{MODEL_NAME}"; os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "all_results.csv" if MODE == "full" else f"rerun_{MODE}.csv")
with open(out_path, "w", newline="") as f: csv.writer(f).writerow(COLUMNS)

rows, n_forecasts, t_start = [], 0, time.monotonic()
for n, (ds_name, term, cfg, key) in enumerate(SELECTED):
    to_univariate = Dataset(name=ds_name, term=term, to_univariate=False).target_dim != 1
    dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
    t0 = time.monotonic(); fc, count = forecasts_for(dataset); n_forecasts += count
    res = evaluate_forecasts(fc, test_data=dataset.test_data, metrics=METRICS, axis=None, batch_size=1024, mask_invalid_label=True,
                             allow_nan_forecast=False, seasonality=get_seasonality(dataset.freq))
    row = [cfg, MODEL_NAME] + [float(res[c.split("/")[1]].iloc[0]) for c in COLUMNS[2:13]] + [dataset_properties_map[key]["domain"], dataset_properties_map[key]["num_variates"]]
    rows.append(row)
    with open(out_path, "a", newline="") as f: csv.writer(f).writerow(row)
    mase, crps = row[5], row[12]; line = f"[{n+1}/{len(SELECTED)}] {cfg}: {count} forecasts in {time.monotonic()-t0:.0f}s | MASE {mase:.4f} CRPS {crps:.4f}"
    if submitted is not None and cfg in submitted.index:
        s = submitted.loc[cfg]; sm, sc = s["eval_metrics/MASE[0.5]"], s["eval_metrics/mean_weighted_sum_quantile_loss"]
        line += f" | submitted MASE {sm:.4f} ({(mase/sm-1)*100:+.1f}%) CRPS {sc:.4f} ({(crps/sc-1)*100:+.1f}%)"
    print(line, flush=True)
elapsed = time.monotonic() - t_start
print(f"wrote {out_path}: {len(rows)} rows, {n_forecasts} forecasts in {elapsed/60:.1f} min")

In [ ]:
# What a full run would take at the throughput just measured (forecast counts of the 97 configurations are known in advance).
FORECASTS_PER_CONFIG = {"bitbrains_fast_storage/5T/long": 4880, "bitbrains_fast_storage/5T/medium": 4880, "bitbrains_fast_storage/5T/short": 43920, "bitbrains_fast_storage/H/short": 4880, "bitbrains_rnd/5T/long": 2000, "bitbrains_rnd/5T/medium": 2000, "bitbrains_rnd/5T/short": 18000, "bitbrains_rnd/H/short": 2000, "bizitobs_application/10S/long": 2, "bizitobs_application/10S/medium": 4, "bizitobs_application/10S/short": 30, "bizitobs_l2c/5T/long": 35, "bizitobs_l2c/5T/medium": 49, "bizitobs_l2c/5T/short": 140, "bizitobs_l2c/H/long": 7, "bizitobs_l2c/H/medium": 7, "bizitobs_l2c/H/short": 42, "bizitobs_service/10S/long": 42, "bizitobs_service/10S/medium": 84, "bizitobs_service/10S/short": 630, "car_parts/M/short": 2509, "covid_deaths/D/short": 266, "electricity/15T/long": 7400, "electricity/15T/medium": 7400, "electricity/15T/short": 7400, "electricity/D/short": 1850, "electricity/H/long": 1850, "electricity/H/medium": 2960, "electricity/H/short": 7400, "electricity/W/short": 1110, "ett1/15T/long": 70, "ett1/15T/medium": 105, "ett1/15T/short": 140, "ett1/D/short": 21, "ett1/H/long": 21, "ett1/H/medium": 28, "ett1/H/short": 140, "ett1/W/short": 14, "ett2/15T/long": 70, "ett2/15T/medium": 105, "ett2/15T/short": 140, "ett2/D/short": 21, "ett2/H/long": 21, "ett2/H/medium": 28, "ett2/H/short": 140, "ett2/W/short": 14, "hierarchical_sales/D/short": 826, "hierarchical_sales/W/short": 472, "hospital/M/short": 767, "jena_weather/10T/long": 168, "jena_weather/10T/medium": 231, "jena_weather/10T/short": 420, "jena_weather/D/short": 42, "jena_weather/H/long": 42, "jena_weather/H/medium": 42, "jena_weather/H/short": 399, "kdd_cup_2018/D/short": 512, "kdd_cup_2018/H/long": 512, "kdd_cup_2018/H/medium": 512, "kdd_cup_2018/H/short": 5120, "loop_seattle/5T/long": 4845, "loop_seattle/5T/medium": 6460, "loop_seattle/5T/short": 6460, "loop_seattle/D/short": 646, "loop_seattle/H/long": 646, "loop_seattle/H/medium": 646, "loop_seattle/H/short": 6137, "m4_daily/D/short": 4227, "m4_hourly/H/short": 414, "m4_monthly/M/short": 48000, "m4_quarterly/Q/short": 24000, "m4_weekly/W/short": 359, "m4_yearly/A/short": 22974, "m_dense/D/short": 90, "m_dense/H/long": 90, "m_dense/H/medium": 120, "m_dense/H/short": 600, "restaurant/D/short": 807, "saugeen/D/short": 20, "saugeen/M/short": 7, "saugeen/W/short": 20, "solar/10T/long": 1096, "solar/10T/medium": 1507, "solar/10T/short": 2740, "solar/D/short": 274, "solar/H/long": 274, "solar/H/medium": 274, "solar/H/short": 2603, "solar/W/short": 137, "sz_taxi/15T/long": 156, "sz_taxi/15T/medium": 156, "sz_taxi/15T/short": 1092, "sz_taxi/H/short": 312, "temperature_rain/D/short": 96216, "us_births/D/short": 20, "us_births/M/short": 2, "us_births/W/short": 14}
total = sum(FORECASTS_PER_CONFIG.values()); done = sum(FORECASTS_PER_CONFIG[cfg] for _, _, cfg, _ in SELECTED)
rate = n_forecasts / elapsed if elapsed else float("nan")
print(f"measured {rate:.1f} forecasts/s; selected configurations = {done:,} of {total:,} forecasts")
print(f"estimated full run: {total / rate / 3600:.1f} h at this rate (set MODE = 'full')")